# World Oil Transit Chokepoints in Plotly

Replica del gráfico `chokepoints` usando Plotly y los Excel de `figure data` publicados por la U.S. Energy Information Administration (EIA). La vista `Total / Ranking` usa los volúmenes 1H25 de la tabla principal del documento EIA.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

import pandas as pd
import plotly.graph_objects as go

START = Path.cwd().resolve()
candidates = [START, *START.parents, Path(__file__).resolve().parent if "__file__" in globals() else START]
PROJECT_ROOT = next((p for p in candidates if (p / "public" / "eia").exists()), START)

DATA_DIR = PROJECT_ROOT / "public" / "eia"
DATA_DIR.mkdir(parents=True, exist_ok=True)
EIA_DOC_URL = "https://www.eia.gov/international/analysis/special-topics/World_Oil_Transit_Chokepoints"
EIA_XLSX_BASE = "https://www.eia.gov/international/content/analysis/special_topics/World_Oil_Transit_Chokepoints/excel"

FIGURES = {
    "Malacca": {"file": "F3.xlsx", "title": "Strait of Malacca"},
    "Hormuz": {"file": "F5.xlsx", "title": "Strait of Hormuz"},
    "Suez + SUMED": {"file": "F7.xlsx", "title": "Suez Canal + SUMED"},
    "Danish Straits": {"file": "F9.xlsx", "title": "Danish Straits"},
    "Turkish Straits": {"file": "F11.xlsx", "title": "Turkish Straits"},
    "Cape of Good Hope": {"file": "F14.xlsx", "title": "Cape of Good Hope"},
}

TOTAL_RANKING_CSV = DATA_DIR / "total_chokepoints_1h25.csv"

def ensure_figure_file(file_name):
    path = DATA_DIR / file_name
    if not path.exists():
        url = f"{EIA_XLSX_BASE}/Chokepoints_FY2026_{file_name}"
        print(f"Downloading {file_name} from EIA...")
        urlretrieve(url, path)
    return path

for meta in FIGURES.values():
    ensure_figure_file(meta["file"])

if not TOTAL_RANKING_CSV.exists():
    TOTAL_RANKING_CSV.write_text(
        "category,value\n"
        "Malacca,23.2\n"
        "Hormuz,20.9\n"
        "Good Hope,9.1\n"
        "Suez + SUMED,4.9\n"
        "Danish Straits,4.9\n"
        "Bab el-Mandeb,4.2\n"
        "Turkish Straits,3.7\n"
        "Panama Canal,2.3\n",
        encoding="utf-8",
    )

In [ ]:
def load_1h25(path: Path, sheet_name):
    df = pd.read_excel(path, sheet_name=sheet_name, header=1)
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "category"})
    df = df[["category", "1H25"]].copy()
    df["category"] = df["category"].astype(str).str.strip()
    df["value"] = pd.to_numeric(df["1H25"], errors="coerce")
    df = df.dropna(subset=["value"])
    df = df[~df["category"].str.lower().str.startswith(("notes", "source", "see"))]
    df = df[df["value"] > 0]
    return df[["category", "value"]].reset_index(drop=True)


def load_chokepoint(key):
    meta = FIGURES[key]
    path = DATA_DIR / meta["file"]
    origin = load_1h25(path, 0)
    destination = load_1h25(path, 1)
    return {
        "key": key,
        "title": meta["title"],
        "origin": origin,
        "destination": destination,
        "origin_total": float(origin["value"].sum()),
        "destination_total": float(destination["value"].sum()),
    }


datasets = {key: load_chokepoint(key) for key in FIGURES}
#datasets["Hormuz"]["origin"].head()
datasets

In [3]:
WIDTH = 1080
HEIGHT = 820
BG = "#050505"
TEXT = "#f2f2f2"
MUTED = "rgba(242,242,242,0.62)"
GREEN = "#16a34a"
RED = "#dc2626"
GOLD = "#ffd166"

FLOW_CX = 450
FLOW_TOP_Y = 118
FLOW_THROAT_Y = 428
FLOW_BOTTOM_Y = 446
FLOW_BOTTOM_END_Y = 748
FLOW_W = 720
FLOW_THROAT_W = 302
SEGMENT_GAP = 18
THROAT_GAP = 0
SCALE = 0.8


def layout_segments(df, total_width, gap):
    total = df["value"].sum()
    usable = total_width - gap * (len(df) - 1)
    x = 0
    out = []
    for _, row in df.iterrows():
        width = max(8, usable * row["value"] / total)
        out.append({"label": row["category"], "value": row["value"], "x": x, "w": width})
        x += width + gap
    return out


def flow_path(top, throat, y0, y1):
    dy = y1 - y0
    c1y = y0 + dy * 0.42
    c2y = y0 + dy * 0.78
    return (
        f"M {top['x']} {y0} "
        f"C {top['x']} {c1y}, {throat['x']} {c2y}, {throat['x']} {y1} "
        f"L {throat['x'] + throat['w']} {y1} "
        f"C {throat['x'] + throat['w']} {c2y}, {top['x'] + top['w']} {c1y}, {top['x'] + top['w']} {y0} Z"
    )


def segmented_flow(df, top_y, bottom_y, invert=False):
    wide_left = FLOW_CX - FLOW_W / 2
    throat_left = FLOW_CX - FLOW_THROAT_W / 2
    wide = layout_segments(df, FLOW_W, SEGMENT_GAP)
    throat = layout_segments(df, FLOW_THROAT_W, THROAT_GAP)
    out = []
    for w, t in zip(wide, throat):
        wide_box = {"x": wide_left + w["x"], "w": w["w"]}
        throat_box = {"x": throat_left + t["x"], "w": t["w"]}
        path = flow_path(throat_box, wide_box, top_y, bottom_y) if invert else flow_path(wide_box, throat_box, top_y, bottom_y)
        out.append({**w, "cx": wide_box["x"] + wide_box["w"] / 2, "path": path})
    return out


def pct_label(value, total):
    return f"{round(value / total * 100):.0f}%"

In [4]:
def add_labels(fig, segments, y, total, above=True):
    for seg in segments:
        label = str(seg["label"])
        if len(label) > 14:
            label = label[:13] + "."
        fig.add_annotation(
            x=seg["cx"],
            y=y,
            text=f"<b>{label}</b><br><b>{pct_label(seg['value'], total)}</b>",
            showarrow=False,
            font=dict(color=TEXT, size=10),
            align="center",
        )


def make_flow_figure(key="Hormuz", mode="percent"):
    if key == "Total / Ranking":
        df = pd.read_csv(TOTAL_RANKING_CSV)
        title = "Total / Ranking"
        center = "World oil flow"
        origin = df
        destination = df
    else:
        data = datasets[key]
        title = data["title"]
        center = title
        origin = data["origin"]
        destination = data["destination"]

    origin_segments = segmented_flow(origin, FLOW_TOP_Y, FLOW_THROAT_Y, invert=False)
    destination_segments = segmented_flow(destination, FLOW_BOTTOM_Y, FLOW_BOTTOM_END_Y, invert=True)

    fig = go.Figure()
    for seg in origin_segments:
        fig.add_shape(type="path", path=seg["path"], fillcolor=GREEN, line=dict(color="rgba(0,0,0,0.8)", width=2))
    for seg in destination_segments:
        fig.add_shape(type="path", path=seg["path"], fillcolor=RED, line=dict(color="rgba(0,0,0,0.8)", width=2))

    fig.add_shape(
        type="rect",
        x0=FLOW_CX - FLOW_THROAT_W / 2 - 12,
        x1=FLOW_CX + FLOW_THROAT_W / 2 + 12,
        y0=FLOW_THROAT_Y - 10,
        y1=FLOW_THROAT_Y + 12,
        fillcolor=BG,
        line=dict(color=BG),
    )
    fig.add_annotation(x=FLOW_CX, y=FLOW_THROAT_Y + 2, text=f"<b>{center}</b>", showarrow=False, font=dict(color="white", size=15))

    add_labels(fig, origin_segments, FLOW_TOP_Y - 34, origin["value"].sum(), above=True)
    add_labels(fig, destination_segments, FLOW_BOTTOM_END_Y + 34, destination["value"].sum(), above=False)

    fig.add_annotation(x=FLOW_CX, y=34, text="<b>Origins</b>", showarrow=False, font=dict(color="white", size=16))
    fig.add_annotation(x=FLOW_CX, y=852, text="<b>Destinations</b>", showarrow=False, font=dict(color="white", size=16))

    fig.update_layout(
        title=f"World Oil Transit Chokepoints, 1H25 · {title}",
        width=WIDTH,
        height=HEIGHT,
        paper_bgcolor=BG,
        plot_bgcolor=BG,
        font=dict(family="Arial", color=TEXT),
        margin=dict(l=20, r=20, t=60, b=20),
        xaxis=dict(range=[0, WIDTH], visible=False),
        yaxis=dict(range=[HEIGHT, 0], visible=False, scaleanchor="x", scaleratio=1),
        showlegend=False,
    )
    return fig


fig = make_flow_figure("Total / Ranking")
fig

In [5]:
# Cambia el key para replicar otro chokepoint:
# "Malacca", "Hormuz", "Suez + SUMED", "Danish Straits", "Turkish Straits", "Cape of Good Hope"
fig = make_flow_figure("Hormuz")
fig